# Session 4: chunking and retrieval experiments

Exploration only. The service depends on `nw/policy`; this notebook imports it.

## What a chunk carries

In [ ]:
from pathlib import Path
from nw.policy.chunking import chunk_corpus
chunks = chunk_corpus(Path("../data/policies"))
c = next(x for x in chunks if x.doc_id == "sla-2025" and "Response" in x.section)
print(c.id, c.effective, c.audience, "current" if c.current else "superseded")
print(c.text[:600])

## Chunk size against chunk count and token spread

In [ ]:
import statistics
for mt in (150, 350, 700, 1200):
    cs = chunk_corpus(Path("../data/policies"), max_tokens=mt)
    toks = [x.tokens for x in cs]
    print(f"max_tokens={mt:5d}: {len(cs):4d} chunks, median {statistics.median(toks):.0f} tokens, max {max(toks)}")

## Dense against lexical against hybrid on one exact-term query

In [ ]:
from nw.policy.retrieval import PolicyIndex, real_embeddings
idx = PolicyIndex.load(Path("../artifacts/policy"), real_embeddings(), Path("../artifacts/policy/chunks.jsonl"))
q = "What does sla-2023 say about service credits?"
for name, hits in [("dense", idx.dense(q, 5)), ("bm25", idx.lexical(q, 5)), ("hybrid", idx.retrieve(q, k=5, rerank=False, current_only=False))]:
    print(name, [(h.chunk.doc_id, h.chunk.section[:22]) for h in hits[:3]])

## Reading the harness output

In [ ]:
import json
r = json.load(open("../artifacts/policy/eval.json"))
agg = r["aggregate"]
print({k: (round(v, 3) if isinstance(v, float) else v) for k, v in agg.items()})
refused_wrong = [x for x in r["results"] if not x["refusal_correct"]]
print(len(refused_wrong), "cases with the wrong refusal decision")
for x in refused_wrong[:5]:
    print(" ", x["id"], "refused" if x["refused"] else "answered", "|", x["answer"][:120])

## Questions to bring back

- Which chunk size gave the best recall, and what did it do to p95 latency?
- For the wrong refusal decisions: was it retrieval, the model, or the citation check?